# Time Series Analysis

The `spectrumCB` function computes a power spectral density (PSD) using **50% overlapping segments** (Welch's method) with linear detrending.

**Algorithm summary:**
1. Split the record into overlapping segments of length `chunk`, stepping forward by `chunk/2`
2. Detrend each segment
3. FFT → one-sided PSD, normalized so that integrating over frequency gives the signal variance
4. Average PSDs across all segments
5. Check Parseval's theorem: `∫ PSD df / var(data) ≈ 1`


## `spectrumCB` function

In [ ]:
import numpy as np
from scipy.signal import detrend as scipy_detrend


def spectrumCB(time, data, chunk):
    """
    Compute a one-sided Power Spectral Density (PSD) using Welch's method:
    50% overlapping segments, linear detrending, FFT, and ensemble averaging.

    Welch's method reduces the variance of the spectral estimate by splitting
    the record into shorter overlapping segments, computing a periodogram for
    each, and averaging them together. The tradeoff is reduced frequency
    resolution (wider spectral peaks) in exchange for a smoother, more stable
    estimate.

    Parameters
    ----------
    time  : array-like
        Time vector in decimal days since the start of the record.
        Used only to compute the mean sample interval (dt).
    data  : array-like
        Data values to be analyzed. Must be NaN-free before calling.
        (Use np.interp or similar to fill gaps beforehand.)
    chunk : int
        Number of samples per segment. Controls the tradeoff between
        frequency resolution (larger chunk → finer resolution) and
        spectral stability (more segments → smoother spectrum).
        Typical choice: divide the record into ~10 equal segments.

    Returns
    -------
    f        : ndarray
        One-sided frequency vector in cycles per day (cpd),
        running from 0 to the Nyquist frequency (1 / 2dt).
    a        : ndarray
        Averaged PSD in [units² / cpd], where units are the units of data.
        Integrating a over f recovers the signal variance.
    parseval : float
        Ratio of the spectral integral to the signal variance.
        Should be close to 1.0; values between 0.95 and 1.05 are acceptable.
    """

    # Ensure inputs are numpy float arrays regardless of what was passed in
    data  = np.asarray(data,  dtype=float)
    time  = np.asarray(time,  dtype=float)
    chunk = int(chunk)

    # -------------------------------------------------------------------------
    # Step 1: Split the record into 50% overlapping segments
    # -------------------------------------------------------------------------
    # Each segment is 'chunk' samples long. The starting index advances by
    # chunk/2 each iteration, so consecutive segments share half their data.
    # This overlap reduces the variance of the averaged PSD estimate.
    segments = []
    step = chunk // 2          # advance by half a segment each time
    ind  = 0
    while ind + chunk <= len(data):
        segments.append(data[ind : ind + chunk])
        ind += step

    # -------------------------------------------------------------------------
    # Step 2: Build the one-sided frequency vector
    # -------------------------------------------------------------------------
    dt = np.nanmean(np.diff(time))      # mean sample interval [days]
    fn = 1.0 / (2.0 * dt)              # Nyquist frequency [cpd] — highest
                                        # resolvable frequency given dt
    N  = chunk                          # samples per segment
    T  = dt * N                         # duration of one segment [days]
    df = 1.0 / T                        # frequency resolution [cpd] — smallest
                                        # distinguishable frequency difference
    f  = np.arange(0, fn + df / 2, df) # one-sided freq vector: 0, df, 2df, ..., fn
    nf = len(f)                         # number of frequency bins (= N//2 + 1)

    # -------------------------------------------------------------------------
    # Step 3: Compute and average the PSD across all segments
    # -------------------------------------------------------------------------
    A = np.empty((len(segments), nf))   # preallocate: rows=segments, cols=freq bins

    for i, seg in enumerate(segments):

        # Remove the best-fit linear trend from the segment.
        # This suppresses spectral leakage caused by a non-zero mean or
        # a slow drift across the segment, which would otherwise bleed
        # energy into neighboring frequency bins.
        seg_dt = scipy_detrend(seg)

        # Compute the full (two-sided) FFT of the detrended segment.
        # fft_vals[0] is the DC (zero-frequency) component.
        # fft_vals[1:N//2] are the positive-frequency components.
        # fft_vals[N//2+1:] are the mirror-image negative-frequency components.
        fft_vals = np.fft.fft(seg_dt)

        # Keep only the one-sided (non-negative frequency) part.
        # Squaring gives power (amplitude²) at each frequency bin.
        amp = np.abs(fft_vals[:nf]) ** 2

        # Normalize by N² to account for the FFT scaling convention.
        # Without this, the FFT output grows with record length.
        amp = amp / N ** 2

        # Multiply by 2 to fold the energy from the negative-frequency side
        # onto the positive-frequency side (one-sided convention).
        # The DC (index 0) and Nyquist (index nf-1) bins are not doubled
        # because they have no negative-frequency counterpart, but the
        # factor-of-2 error there is negligible in practice.
        amp = amp * 2

        # Divide by df to convert from amplitude² per bin to a density:
        # PSD [units² / cpd]. This normalization ensures that integrating
        # the PSD over frequency returns the signal variance (Parseval).
        amp = amp / df

        A[i] = amp

    # Average the PSD across all segments to reduce variance
    a = A.mean(axis=0)

    # -------------------------------------------------------------------------
    # Step 4: Parseval's theorem check
    # -------------------------------------------------------------------------
    # Parseval's theorem states that the total power in the time domain
    # (variance of the signal) must equal the integral of the PSD over frequency.
    # We compute this ratio as a sanity check. A value near 1.0 confirms
    # that the normalization is self-consistent.
    variance  = np.nanstd(data) ** 2
    int_spec  = np.trapezoid(a, f)          # ∫ PSD df  [units²]
    parseval  = int_spec / variance         # should be ≈ 1.00

    print(f"Segments used:  {len(segments)}")
    print(f"Parseval check: ∫PSD df / var(data) = {parseval:.4f}  (ideal = 1.00)")

    return f, a, parseval


---
## Demo 1 — CLASS10 mooring velocity

A ~333-day current velocity record sampled at 30 minutes. Reference lines mark key physical frequencies:

- 1 cpd — diurnal tidal band (K₁)
- 24/12.42 ≈ 1.93 cpd — semidiurnal M₂ tide
- 24/12.00 = 2.00 cpd — semidiurnal S₂ tide
- Local inertial / Coriolis frequency (La Jolla)


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# =============================================================================
# Load the CLASS10 mooring velocity data
# =============================================================================
# The data are stored as a JSON file with two keys:
#   'time'     — list of ISO 8601 timestamp strings (e.g. '2021-03-15T00:30')
#   'data'     — list of velocity values in m/s
#   'metadata' — dict with start/end dates, sample count, etc.

with open('sample_data.json') as f:
    sd = json.load(f)

data = np.array(sd['data'])   # velocity time series [m/s]

# -------------------------------------------------------------------------
# Convert ISO timestamp strings to decimal days since the record start.
# -------------------------------------------------------------------------
# spectrumCB expects time in days (used only to compute dt), so we:
#   1. Parse each string into a Python datetime object
#   2. Subtract the first timestamp to get a timedelta
#   3. Convert to total seconds, then divide by 86400 to get days
t_parsed = [datetime.strptime(s, '%Y-%m-%dT%H:%M') for s in sd['time']]
t0       = t_parsed[0]                        # reference epoch (record start)
time     = np.array(
    [(t - t0).total_seconds() / 86400.0 for t in t_parsed]
)  # [decimal days]

print(f"Record:  {sd['metadata']['start']}  →  {sd['metadata']['end']}")
print(f"Samples: {sd['metadata']['n_samples']:,}  |  dt: {np.nanmean(np.diff(time))*24:.2f} hours")

# =============================================================================
# Compute the power spectral density
# =============================================================================
# We divide the record into 10 equal chunks. Each chunk is ~33 days long,
# giving a frequency resolution of ~1/33 ≈ 0.03 cpd — fine enough to
# clearly resolve the diurnal (1 cpd) and semidiurnal (2 cpd) tidal peaks,
# which are separated by ~0.07 cpd.
num_chunks = 10
chunk_len  = len(data) // num_chunks   # samples per segment

f, psd, parseval = spectrumCB(time, data, chunk_len)

# =============================================================================
# Plot the PSD on a log-log scale with tidal reference lines
# =============================================================================
fig, ax = plt.subplots(figsize=(9, 5))

# Plot PSD — skip the zero-frequency bin (index 0) to avoid log(0) issues
ax.loglog(f[1:], psd[1:], color='steelblue', lw=1.2, label='PSD')

# -------------------------------------------------------------------------
# Tidal and dynamical reference frequencies
# -------------------------------------------------------------------------
# Each vertical dashed line marks a known physical frequency:
#   K₁   — principal lunar/solar diurnal constituent    (period = 23.93 h)
#   M₂   — principal lunar semidiurnal constituent      (period = 12.42 h)
#   S₂   — principal solar semidiurnal constituent      (period = 12.00 h)
#   f_Cor — local inertial (Coriolis) frequency at La Jolla (~32.7°N)
#            f = 2Ω sin(lat) ≈ 7.83×10⁻⁵ rad/s → period ≈ 22.3 h
#            Expressed in cpd: 24 / 22.3 ≈ 1.08 cpd (using 15.566 h → 24/15.566)
ref_lines = [
    (1,            'K₁  (1 cpd)',       'orange'),
    (24 / 12.42,   'M₂  (1.93 cpd)',   'crimson'),
    (24 / 12.00,   'S₂  (2.00 cpd)',   'firebrick'),
    (24 / 15.566,  'f_Cor (La Jolla)', 'seagreen'),
]
for freq, label, color in ref_lines:
    ax.axvline(freq, color=color, ls='--', lw=1, label=label)

# Axis limits, labels, and formatting
ax.set_ylim([1e-6, 1e-1])
ax.set_xlabel('Frequency [cpd]')
ax.set_ylabel(r'$\Phi_v$ [(m/s)² / cpd]')
ax.set_title('CLASS10 Mooring — Velocity Power Spectral Density')
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.3)   # grid on both major and minor ticks
plt.tight_layout()
plt.show()


---
## Demo 2 — La Jolla tide gauge

`la_jolla_tide.json` contains the **full** NOAA La Jolla tide gauge record (~100 years, hourly, in mm). Missing values are stored as `null` and are linearly interpolated before spectral analysis.


In [ ]:
from datetime import datetime

# =============================================================================
# Load the La Jolla NOAA tide gauge record
# =============================================================================
# la_jolla_tide.json contains the full ~100-year record (1925–2025), sampled
# hourly. Sea level is stored in millimetres relative to a fixed datum.
# Missing observations are stored as JSON null, which Python reads as None.

with open('la_jolla_tide.json') as f:
    lj = json.load(f)

# Replace None (JSON null) with np.nan so NumPy handles missing values cleanly.
# We keep NaNs in the array for now and interpolate them in the next step.
sea_level = np.array(
    [np.nan if v is None else v for v in lj['sea_level']]
)  # [mm]

# -------------------------------------------------------------------------
# Convert ISO timestamp strings to decimal days since the record start
# -------------------------------------------------------------------------
# Same approach as Demo 1: parse → timedelta → total seconds → days.
t_parsed = [datetime.strptime(s, '%Y-%m-%dT%H:%M') for s in lj['time']]
t0       = t_parsed[0]
time_lj  = np.array(
    [(t - t0).total_seconds() / 86400.0 for t in t_parsed]
)  # [decimal days]

dt_lj = np.nanmean(np.diff(time_lj))   # mean sample interval [days]; should be 1/24

print(f"Record:  {lj['metadata']['start']}  →  {lj['metadata']['end']}")
print(f"Samples: {len(sea_level):,}  |  Missing: {lj['metadata']['n_nan']:,}  ({lj['metadata']['n_nan']/len(sea_level)*100:.1f}%)")
print(f"dt:      {dt_lj*24:.4f} hours")

# =============================================================================
# Linearly interpolate across missing values (NaNs)
# =============================================================================
# spectrumCB requires a NaN-free input. We use np.interp, which performs
# piecewise linear interpolation. The strategy:
#   - Build a "known-good" subset by masking out NaN positions
#   - Evaluate a linear interpolant at ALL time points (including the gaps)
# This is appropriate for short isolated gaps. The ~5.5% missing fraction
# here is spread throughout the record, so linear interpolation is reasonable
# — it will not significantly bias the spectral estimates at tidal frequencies.
nan_mask         = np.isnan(sea_level)              # True where data is missing
sea_level_interp = np.interp(
    time_lj,                    # x values to evaluate at (all times)
    time_lj[~nan_mask],         # x values with known data
    sea_level[~nan_mask]        # corresponding known sea-level values
)

print(f"Sea level range: {sea_level_interp.min():.0f} – {sea_level_interp.max():.0f} mm (after interpolation)")


---
## Quick look — raw tidal signal

Plot a 30-day window to visually confirm the tidal variability before computing the spectrum.


In [ ]:
# =============================================================================
# Quick look — first 30 days of the tidal record
# =============================================================================
# Before computing any spectra, it is good practice to visually inspect the
# raw signal. This plot confirms:
#   - The tidal signal looks physically reasonable (semi-diurnal pattern visible)
#   - The interpolation has not introduced obvious artifacts
#   - The amplitude and baseline are consistent with expected sea-level values
#
# 30 days is long enough to see the spring-neap cycle (~14.8 days) driven by
# the interference between M₂ and S₂, while remaining readable on screen.

nplot = int(30 / dt_lj)    # number of hourly samples in 30 days (= 720)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time_lj[:nplot], sea_level_interp[:nplot], lw=0.8, color='navy')
ax.set_xlabel('Time [days from record start]')
ax.set_ylabel('Sea level [mm]')
ax.set_title('La Jolla Tide Gauge — first 30 days (interpolated)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Compute the tidal power spectral density
# =============================================================================
# We again divide the full record into 10 equal segments.
# With ~100 years of hourly data (878,016 samples), each segment is
# ~87,800 samples ≈ 10 years long.
#
# Frequency resolution: df = 1 / (chunk_lj * dt_lj)
#   = 1 / (87800 samples × 1/24 days/sample)
#   ≈ 1 / 3658 days  ≈ 2.7 × 10⁻⁴ cpd
#
# This extremely fine resolution means the narrow tidal peaks (K₁, M₂, S₂)
# are well resolved and clearly separated from neighboring constituents.
# The 19 overlapping segments (from 50% overlap of 10 chunks) provide a
# smooth, stable spectral estimate across the full frequency range.

num_chunks_lj = 10
chunk_lj      = len(sea_level_interp) // num_chunks_lj   # samples per segment

f_lj, psd_lj, parseval_lj = spectrumCB(time_lj, sea_level_interp, chunk_lj)


In [ ]:
# =============================================================================
# Plot the tidal power spectral density
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 5))

# Plot PSD — skip index 0 (zero frequency / DC offset) to avoid log(0)
ax.loglog(f_lj[1:], psd_lj[1:], color='navy', lw=1.2, label='PSD')

# -------------------------------------------------------------------------
# Reference tidal constituents
# -------------------------------------------------------------------------
# The five most energetic constituents in the La Jolla record:
#
#   O₁  — principal lunar diurnal        period = 25.82 h  (0.93 cpd)
#   K₁  — lunisolar diurnal              period = 23.93 h  (1.00 cpd)
#   M₂  — principal lunar semidiurnal    period = 12.42 h  (1.93 cpd)
#   S₂  — principal solar semidiurnal    period = 12.00 h  (2.00 cpd)
#   M₄  — first overtide of M₂           period =  6.21 h  (3.87 cpd)
#          Generated by nonlinear shallow-water dynamics; energy is
#          transferred from M₂ to M₄ in shallow coastal regions.
#
# The California coast has a mixed, mainly semidiurnal tidal regime:
# M₂ and S₂ dominate, but the diurnal inequality (difference between
# the two daily highs and two daily lows) is significant — hence the
# visible O₁ and K₁ peaks.
tidal_lines = [
    (24 / 25.82,  'O₁  (0.93 cpd)',   'orange'),
    (1.0,         'K₁  (1.00 cpd)',   'darkorange'),
    (24 / 12.42,  'M₂  (1.93 cpd)',   'crimson'),
    (24 / 12.00,  'S₂  (2.00 cpd)',   'firebrick'),
    (24 / 6.21,   'M₄  (3.87 cpd)',   'purple'),
]
for freq, label, color in tidal_lines:
    ax.axvline(freq, color=color, ls='--', lw=1, label=label)

# Axis labels, legend, and grid
ax.set_xlabel('Frequency [cpd]')
ax.set_ylabel(r'$\Phi_\eta$ [mm² / cpd]')
ax.set_title('La Jolla Tide Gauge — Power Spectral Density')
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.3)   # minor grid helps read log axes
plt.tight_layout()
plt.show()
